# DWS SnowCamp — Hands-On Lab

## Building an End-to-End Client Reporting Data Product on Snowflake

**Duration**: 1.5 hours | **Level**: Intermediate | **Account**: Snowflake Trial

---

Welcome to the DWS SnowCamp hands-on lab! In this session you will build a complete data product pipeline for **asset management client reporting** — from raw synthetic data through to a published, governed data product with an interactive dashboard.

| Part | Topic | Duration |
|------|-------|----------|
| 1 | Environment Setup & Synthetic Data | 20 min |
| 2 | dbt Transformation (Staging → Marts) | 25 min |
| 3 | Streamlit Dashboard | 20 min |
| 4 | Internal Marketplace Listing | 10 min |
| 5 | Platform Administration & Monitoring | 15 min |

> **Reference**: [Snowflake Notebooks Documentation](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks)

### Architecture Overview

In this lab you will build the following pipeline inside your trial account:

```
Source Data (Python)  →  RAW schema  →  ANALYTICS schema  →  MARTS schema  →  Streamlit App
                          (Bronze)        (Silver)              (Gold)           ↓
                                                                       Marketplace Listing
```

**Three-Layer Architecture** (mirrors the DWS production design):

| Layer | Schema | Purpose |
|-------|--------|---------|
| Bronze / Raw | `RAW` | Synthetic data mimicking on-prem sources |
| Silver / Curated | `ANALYTICS` | Staging views and intermediate tables |
| Gold / Marts | `MARTS` | Business-ready fact and dimension tables |

> In a production DWS deployment, raw data arrives via [OpenFlow](https://docs.snowflake.com/en/user-guide/data-load-openflow) from on-prem Oracle and SQL Server databases. Today we generate synthetic data instead.

---
## Part 1: Environment Setup & Synthetic Data (20 min)

First, let's create the database, schemas, and warehouse in your trial account. Then we'll generate synthetic asset-management data that mirrors a real client-reporting dataset.

> **Reference**: [CREATE DATABASE](https://docs.snowflake.com/en/sql-reference/sql/create-database) | [CREATE WAREHOUSE](https://docs.snowflake.com/en/sql-reference/sql/create-warehouse) | [CREATE SCHEMA](https://docs.snowflake.com/en/sql-reference/sql/create-schema)

In [ ]:
-- =============================================================
-- Create the lab database with three schemas
-- matching the DWS three-layer architecture
-- =============================================================

CREATE DATABASE IF NOT EXISTS SNOWCAMP_LAB;

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.RAW
    COMMENT = 'Raw/Bronze layer - synthetic source data';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.ANALYTICS
    COMMENT = 'Staging and intermediate layer - cleaned and enriched';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.MARTS
    COMMENT = 'Gold/Mart layer - business-ready data products';

-- Create a compute warehouse for the lab
CREATE WAREHOUSE IF NOT EXISTS WH_LAB
    WAREHOUSE_SIZE   = 'XSMALL'
    AUTO_SUSPEND     = 60
    AUTO_RESUME      = TRUE
    COMMENT          = 'SnowCamp hands-on lab warehouse';

-- Enable cross-region inference for Cortex AI features
-- https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

-- Set session context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WH_LAB;
USE DATABASE SNOWCAMP_LAB;
USE SCHEMA RAW;

SELECT 'Environment ready!' AS status;

### Data Model

We will generate six tables representing a typical asset-management data landscape:

| Table | Description | Approx. Rows |
|-------|-------------|-------------|
| `CLIENTS` | Institutional clients (pension funds, insurance, etc.) | 30 |
| `PORTFOLIOS` | Investment portfolios linked to clients | 100 |
| `SECURITIES` | Tradeable instruments (equities, bonds, ETFs) | 200 |
| `HOLDINGS` | Daily position snapshots per portfolio | ~5,000 |
| `TRANSACTIONS` | Trade executions (buys and sells) | 10,000 |
| `BENCHMARKS` | Daily returns for 5 benchmark indices | ~1,200 |

> **Reference**: [Snowpark Python Developer Guide](https://docs.snowflake.com/en/developer-guide/snowpark/python/index)

In [ ]:
# ================================================================
# Generate synthetic asset-management data and load into RAW schema
# ================================================================
import pandas as pd
import random
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session

session = get_active_session()
random.seed(42)

# --- Configuration ---
NUM_CLIENTS       = 30
NUM_PORTFOLIOS    = 100
NUM_SECURITIES    = 200
NUM_HOLDING_DAYS  = 20
NUM_TRANSACTIONS  = 10000
BENCHMARK_DAYS    = 252

# --- Reference data ---
REGIONS         = ['EMEA', 'APAC', 'Americas']
AUM_TIERS       = ['Tier 1 (>1B)', 'Tier 2 (500M-1B)', 'Tier 3 (<500M)']
PORTFOLIO_TYPES = ['Equity', 'Fixed Income', 'Multi-Asset', 'ESG', 'Alternatives']
SECTORS         = ['Technology', 'Healthcare', 'Financials', 'Energy',
                   'Consumer Discretionary', 'Industrials', 'Materials',
                   'Utilities', 'Real Estate', 'Communication Services']
ASSET_CLASSES   = ['Equity', 'Government Bond', 'Corporate Bond', 'ETF', 'Money Market']
CURRENCIES      = ['EUR', 'USD', 'GBP', 'CHF', 'JPY']
BENCHMARKS      = {'BM001': 'MSCI World', 'BM002': 'Euro Stoxx 50',
                   'BM003': 'S&P 500', 'BM004': 'Bloomberg Global Agg',
                   'BM005': 'FTSE 100'}
FIRST_NAMES     = ['Anna', 'Hans', 'Maria', 'Klaus', 'Sophie', 'Thomas',
                   'Emma', 'Felix', 'Laura', 'Max', 'Julia', 'Stefan',
                   'Lisa', 'Andreas', 'Sarah']
LAST_NAMES      = ['Mueller', 'Schmidt', 'Fischer', 'Weber', 'Meyer',
                   'Wagner', 'Becker', 'Schulz', 'Hoffmann', 'Koch',
                   'Richter', 'Klein', 'Wolf', 'Braun', 'Lange']
COMPANY_SUFFIX  = ['Capital', 'Invest', 'Advisors', 'Partners',
                   'Asset Mgmt', 'Wealth', 'Fund Services']

# ---- 1. CLIENTS ----
clients = []
for i in range(1, NUM_CLIENTS + 1):
    name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)} {random.choice(COMPANY_SUFFIX)}"
    clients.append({
        'CLIENT_ID': f'CLT{i:04d}',
        'CLIENT_NAME': name,
        'REGION': random.choice(REGIONS),
        'AUM_TIER': random.choice(AUM_TIERS),
        'ONBOARDING_DATE': (datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1500))).strftime('%Y-%m-%d')
    })

# ---- 2. PORTFOLIOS ----
portfolios = []
for i in range(1, NUM_PORTFOLIOS + 1):
    client = random.choice(clients)
    portfolios.append({
        'PORTFOLIO_ID': f'PF{i:04d}',
        'CLIENT_ID': client['CLIENT_ID'],
        'PORTFOLIO_NAME': f"{client['CLIENT_NAME'].split()[0]} {random.choice(PORTFOLIO_TYPES)} Fund {i}",
        'PORTFOLIO_TYPE': random.choice(PORTFOLIO_TYPES),
        'BENCHMARK_ID': random.choice(list(BENCHMARKS.keys())),
        'INCEPTION_DATE': (datetime(2019, 1, 1) + timedelta(days=random.randint(0, 1800))).strftime('%Y-%m-%d')
    })

# ---- 3. SECURITIES ----
securities = []
prefixes = ['DE', 'US', 'GB', 'FR', 'JP', 'CH']
adjectives = ['Global', 'European', 'American', 'Asian', 'Pacific', 'Emerging']
suffixes = ['Fund', 'ETF', 'Note', 'Bond', 'Trust', 'Index']
for i in range(1, NUM_SECURITIES + 1):
    securities.append({
        'SECURITY_ID': f'SEC{i:04d}',
        'ISIN': f'{random.choice(prefixes)}{random.randint(1000000000, 9999999999)}',
        'SECURITY_NAME': f'{random.choice(adjectives)} {random.choice(SECTORS)} {random.choice(suffixes)} {i}',
        'SECTOR': random.choice(SECTORS),
        'ASSET_CLASS': random.choice(ASSET_CLASSES),
        'CURRENCY': random.choice(CURRENCIES)
    })

# ---- 4. HOLDINGS ----
holdings = []
hid = 1
base_date = datetime(2025, 1, 6)
for day_offset in range(NUM_HOLDING_DAYS):
    d = base_date + timedelta(days=day_offset)
    if d.weekday() >= 5:
        continue
    for pf in random.sample(portfolios, min(50, len(portfolios))):
        for sec in random.sample(securities, random.randint(3, 10)):
            qty = random.randint(100, 50000)
            price = round(random.uniform(10, 500), 2)
            holdings.append({
                'HOLDING_ID': hid,
                'PORTFOLIO_ID': pf['PORTFOLIO_ID'],
                'SECURITY_ID': sec['SECURITY_ID'],
                'AS_OF_DATE': d.strftime('%Y-%m-%d'),
                'QUANTITY': qty,
                'MARKET_VALUE': round(qty * price, 2)
            })
            hid += 1

# ---- 5. TRANSACTIONS ----
transactions = []
for i in range(1, NUM_TRANSACTIONS + 1):
    pf = random.choice(portfolios)
    sec = random.choice(securities)
    side = random.choice(['BUY', 'SELL'])
    qty = random.randint(50, 10000)
    price = round(random.uniform(10, 500), 2)
    td = base_date + timedelta(days=random.randint(0, NUM_HOLDING_DAYS - 1))
    transactions.append({
        'TRANSACTION_ID': f'TXN{i:06d}',
        'PORTFOLIO_ID': pf['PORTFOLIO_ID'],
        'SECURITY_ID': sec['SECURITY_ID'],
        'TRADE_DATE': td.strftime('%Y-%m-%d'),
        'SIDE': side,
        'QUANTITY': qty,
        'PRICE': price,
        'AMOUNT': round(qty * price, 2)
    })

# ---- 6. BENCHMARKS ----
benchmarks = []
for bm_id, bm_name in BENCHMARKS.items():
    for day_offset in range(BENCHMARK_DAYS):
        d = datetime(2024, 2, 1) + timedelta(days=day_offset)
        if d.weekday() >= 5:
            continue
        benchmarks.append({
            'BENCHMARK_ID': bm_id,
            'BENCHMARK_NAME': bm_name,
            'AS_OF_DATE': d.strftime('%Y-%m-%d'),
            'DAILY_RETURN': round(random.gauss(0.0003, 0.012), 6)
        })

# ---- Write to Snowflake ----
tables = {
    'CLIENTS': pd.DataFrame(clients),
    'PORTFOLIOS': pd.DataFrame(portfolios),
    'SECURITIES': pd.DataFrame(securities),
    'HOLDINGS': pd.DataFrame(holdings),
    'TRANSACTIONS': pd.DataFrame(transactions),
    'BENCHMARKS': pd.DataFrame(benchmarks),
}

for name, df in tables.items():
    session.write_pandas(df, name, auto_create_table=True, overwrite=True,
                         database='SNOWCAMP_LAB', schema='RAW')
    print(f"  {name}: {len(df):,} rows")

print("\nAll synthetic data loaded into SNOWCAMP_LAB.RAW!")

In [ ]:
-- Validate the raw data
SELECT 'CLIENTS'      AS table_name, COUNT(*) AS row_count FROM SNOWCAMP_LAB.RAW.CLIENTS
UNION ALL
SELECT 'PORTFOLIOS',   COUNT(*) FROM SNOWCAMP_LAB.RAW.PORTFOLIOS
UNION ALL
SELECT 'SECURITIES',   COUNT(*) FROM SNOWCAMP_LAB.RAW.SECURITIES
UNION ALL
SELECT 'HOLDINGS',     COUNT(*) FROM SNOWCAMP_LAB.RAW.HOLDINGS
UNION ALL
SELECT 'TRANSACTIONS', COUNT(*) FROM SNOWCAMP_LAB.RAW.TRANSACTIONS
UNION ALL
SELECT 'BENCHMARKS',   COUNT(*) FROM SNOWCAMP_LAB.RAW.BENCHMARKS
ORDER BY table_name;

In [ ]:
-- Quick look at the data
SELECT * FROM SNOWCAMP_LAB.RAW.CLIENTS    LIMIT 5;
SELECT * FROM SNOWCAMP_LAB.RAW.HOLDINGS   LIMIT 5;
SELECT * FROM SNOWCAMP_LAB.RAW.BENCHMARKS LIMIT 5;

---
## Part 2: dbt Transformation (25 min)

Now we will transform the raw data into analytics-ready tables using the **three-layer pattern** that dbt Projects on Snowflake automate:

| Layer | Purpose | Materialisation |
|-------|---------|-----------------|
| **Staging** | Clean, rename, type-cast raw sources | Views |
| **Intermediate** | Join, enrich, calculate | Tables |
| **Marts** | Business-ready fact & dimension tables | Tables |

We will first run the transformations manually with SQL so you understand what each layer does. Then we will deploy the same logic as a **dbt Project on Snowflake**.

> **Reference**: [dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) | [CREATE VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-view)

### 2a. Staging Layer (Views)

Staging models sit on top of raw tables. They **clean, rename, and type-cast** columns, but do not join across sources. We materialise them as views so they always reflect the latest raw data.

In [ ]:
USE SCHEMA SNOWCAMP_LAB.ANALYTICS;

-- Staging: Holdings
CREATE OR REPLACE VIEW STG_HOLDINGS AS
SELECT
    holding_id,
    portfolio_id,
    security_id,
    as_of_date,
    quantity,
    market_value,
    CASE WHEN quantity > 0 THEN ROUND(market_value / quantity, 4) ELSE 0 END AS unit_price
FROM SNOWCAMP_LAB.RAW.HOLDINGS
WHERE market_value IS NOT NULL;

-- Staging: Transactions
CREATE OR REPLACE VIEW STG_TRANSACTIONS AS
SELECT
    transaction_id,
    portfolio_id,
    security_id,
    trade_date,
    side,
    quantity,
    price,
    amount,
    CASE side WHEN 'BUY' THEN quantity WHEN 'SELL' THEN -quantity ELSE 0 END AS signed_quantity
FROM SNOWCAMP_LAB.RAW.TRANSACTIONS
WHERE price > 0;

-- Staging: Securities
CREATE OR REPLACE VIEW STG_SECURITIES AS
SELECT
    security_id,
    isin,
    security_name,
    sector,
    asset_class,
    currency
FROM SNOWCAMP_LAB.RAW.SECURITIES;

SELECT 'Staging views created!' AS status;

### 2b. Intermediate Layer (Tables)

Intermediate models **join across staging models** and calculate derived metrics. Here we join holdings with securities to get enriched positions with portfolio weights.

In [ ]:
-- Intermediate: Portfolio positions enriched with security details and weights
CREATE OR REPLACE TABLE SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS AS
WITH portfolio_totals AS (
    SELECT portfolio_id, as_of_date, SUM(market_value) AS total_portfolio_value
    FROM STG_HOLDINGS
    GROUP BY portfolio_id, as_of_date
)
SELECT
    h.holding_id,
    h.portfolio_id,
    h.security_id,
    h.as_of_date,
    h.quantity,
    h.market_value,
    h.unit_price,
    s.isin,
    s.security_name,
    s.sector,
    s.asset_class,
    s.currency,
    pt.total_portfolio_value,
    CASE
        WHEN pt.total_portfolio_value > 0
        THEN ROUND(h.market_value / pt.total_portfolio_value, 6)
        ELSE 0
    END AS portfolio_weight
FROM STG_HOLDINGS h
JOIN STG_SECURITIES s ON h.security_id = s.security_id
JOIN portfolio_totals pt ON h.portfolio_id = pt.portfolio_id AND h.as_of_date = pt.as_of_date;

SELECT COUNT(*) AS intermediate_rows FROM SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS;

### 2c. Mart Layer (Tables)

Mart tables are **business-ready data products**. We create:

- `F_POSITIONS_DAILY` — Fact table: daily holdings with full context
- `F_PERFORMANCE_DAILY` — Fact table: daily portfolio returns vs benchmark
- `D_PORTFOLIO` — Dimension: portfolio attributes with client info
- `D_CLIENT` — Dimension: client master data

> **Reference**: [CREATE TABLE AS SELECT](https://docs.snowflake.com/en/sql-reference/sql/create-table)

In [ ]:
USE SCHEMA SNOWCAMP_LAB.MARTS;

-- Fact: Daily Positions
CREATE OR REPLACE TABLE F_POSITIONS_DAILY AS
SELECT
    p.holding_id,
    p.portfolio_id,
    po.client_id,
    po.portfolio_name,
    po.portfolio_type,
    po.benchmark_id,
    p.security_id,
    p.isin,
    p.security_name,
    p.sector,
    p.asset_class,
    p.currency,
    p.as_of_date,
    p.quantity,
    p.market_value,
    p.unit_price,
    p.total_portfolio_value,
    p.portfolio_weight
FROM SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS p
JOIN SNOWCAMP_LAB.RAW.PORTFOLIOS po ON p.portfolio_id = po.portfolio_id;

-- Fact: Daily Performance
CREATE OR REPLACE TABLE F_PERFORMANCE_DAILY AS
WITH portfolio_values AS (
    SELECT portfolio_id, as_of_date, SUM(market_value) AS total_market_value
    FROM SNOWCAMP_LAB.ANALYTICS.STG_HOLDINGS
    GROUP BY portfolio_id, as_of_date
),
portfolio_returns AS (
    SELECT
        portfolio_id, as_of_date, total_market_value,
        LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date) AS prev_mv,
        CASE
            WHEN LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date) > 0
            THEN ROUND((total_market_value - LAG(total_market_value)
                 OVER (PARTITION BY portfolio_id ORDER BY as_of_date))
                 / LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date), 6)
            ELSE 0
        END AS daily_return
    FROM portfolio_values
)
SELECT
    pr.portfolio_id,
    pf.client_id,
    pf.benchmark_id,
    pr.as_of_date,
    pr.total_market_value,
    pr.daily_return  AS portfolio_return,
    COALESCE(b.daily_return, 0) AS benchmark_return,
    ROUND(pr.daily_return - COALESCE(b.daily_return, 0), 6) AS active_return
FROM portfolio_returns pr
JOIN SNOWCAMP_LAB.RAW.PORTFOLIOS pf ON pr.portfolio_id = pf.portfolio_id
LEFT JOIN SNOWCAMP_LAB.RAW.BENCHMARKS b ON pf.benchmark_id = b.benchmark_id AND pr.as_of_date = b.as_of_date
WHERE pr.prev_mv IS NOT NULL;

-- Dimension: Portfolio
CREATE OR REPLACE TABLE D_PORTFOLIO AS
SELECT
    p.portfolio_id, p.portfolio_name, p.portfolio_type,
    p.benchmark_id, p.inception_date,
    p.client_id, c.client_name, c.region, c.aum_tier
FROM SNOWCAMP_LAB.RAW.PORTFOLIOS p
JOIN SNOWCAMP_LAB.RAW.CLIENTS c ON p.client_id = c.client_id;

-- Dimension: Client
CREATE OR REPLACE TABLE D_CLIENT AS
SELECT client_id, client_name, region, aum_tier, onboarding_date
FROM SNOWCAMP_LAB.RAW.CLIENTS;

SELECT 'Mart tables created!' AS status;

In [ ]:
-- Explore the mart: AUM by region
SELECT c.region,
       COUNT(DISTINCT f.portfolio_id) AS portfolios,
       ROUND(SUM(f.market_value), 0) AS total_aum
FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
WHERE f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
GROUP BY c.region
ORDER BY total_aum DESC;

### 2d. dbt Projects on Snowflake

The SQL above is exactly what **dbt** automates. With dbt Projects on Snowflake, you get version-controlled models, automated tests, lineage, and scheduling — all inside Snowsight with no external infrastructure.

This lab’s GitHub repo contains a full dbt project at `dbt/snowcamp_client_reporting/` with the same staging, intermediate, and mart models you just ran manually.

Below we create a **Git Repository integration**, import the dbt project, and execute a `dbt build`.

> **Reference**: [dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) | [Git Repository Integration](https://docs.snowflake.com/en/developer-guide/git/git-setting-up) | [CREATE API INTEGRATION (Git)](https://docs.snowflake.com/en/sql-reference/sql/create-api-integration-git)

In [ ]:
-- Step 1: Create an API integration for GitHub
CREATE OR REPLACE API INTEGRATION snowcamp_git_api
    API_PROVIDER       = git_https_api
    API_ALLOWED_PREFIXES = ('https://github.com/')
    ENABLED            = TRUE;

-- Step 2: Create a Git Repository pointing to the lab repo
CREATE OR REPLACE GIT REPOSITORY SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO
    API_INTEGRATION = snowcamp_git_api
    ORIGIN          = 'https://github.com/sfc-gh-epolano/dws-snowcamp-lab.git';

-- Step 3: Fetch the latest content
ALTER GIT REPOSITORY SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO FETCH;

-- Step 4: Verify the repo contents
SHOW GIT BRANCHES IN SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO;

> **Note**: The `CREATE DBT PROJECT` and `EXECUTE DBT PROJECT` commands require the dbt Projects feature to be enabled in your account. If these commands are not yet available in your trial, the manual SQL transformations above have already built the same tables.
>
> ```sql
> -- When available, run:
> CREATE OR REPLACE DBT PROJECT SNOWCAMP_LAB.RAW.SNOWCAMP_CLIENT_REPORTING
>     FROM GIT REPOSITORY SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO
>     USING 'dbt/snowcamp_client_reporting';
>
> EXECUTE DBT PROJECT SNOWCAMP_LAB.RAW.SNOWCAMP_CLIENT_REPORTING
>     COMMAND 'build'
>     WAREHOUSE = 'WH_LAB';
> ```

---
## Part 3: Streamlit Dashboard (20 min)

Now let’s turn the mart tables into an **interactive client-reporting dashboard** using **Streamlit in Snowflake**. No infrastructure to manage — the app runs inside your Snowflake account, governed by RBAC.

In a Snowflake Notebook, you can render Streamlit components directly in Python cells. We’ll build the dashboard right here, then show how to deploy it as a standalone app.

> **Reference**: [Streamlit in Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) | [Streamlit API Reference](https://docs.streamlit.io/develop/api-reference)

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("DWS Client Reporting Dashboard")
st.caption("Built with Streamlit in Snowflake | SnowCamp Hands-On Lab")

# ---- Sidebar Filters ----
st.sidebar.header("Filters")

regions = session.sql(
    "SELECT DISTINCT region FROM SNOWCAMP_LAB.MARTS.D_CLIENT ORDER BY 1"
).to_pandas()["REGION"].tolist()
selected_region = st.sidebar.multiselect("Region", regions, default=regions)

asset_classes = session.sql(
    "SELECT DISTINCT asset_class FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY ORDER BY 1"
).to_pandas()["ASSET_CLASS"].tolist()
selected_ac = st.sidebar.multiselect("Asset Class", asset_classes, default=asset_classes)

region_filter = ", ".join([f"'{r}'" for r in selected_region]) if selected_region else "''"
ac_filter = ", ".join([f"'{a}'" for a in selected_ac]) if selected_ac else "''"

# ---- KPI Metrics ----
kpi = session.sql(f"""
    SELECT
        COUNT(DISTINCT f.client_id)    AS clients,
        COUNT(DISTINCT f.portfolio_id) AS portfolios,
        ROUND(SUM(f.market_value), 0)  AS aum
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE c.region IN ({region_filter})
      AND f.asset_class IN ({ac_filter})
      AND f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
""").to_pandas()

c1, c2, c3 = st.columns(3)
c1.metric("Clients", f"{kpi['CLIENTS'].iloc[0]:,}")
c2.metric("Portfolios", f"{kpi['PORTFOLIOS'].iloc[0]:,}")
c3.metric("Total AUM", f"${kpi['AUM'].iloc[0]:,.0f}")

# ---- AUM by Region (bar chart) ----
st.subheader("AUM by Region")
aum_region = session.sql(f"""
    SELECT c.region, ROUND(SUM(f.market_value), 0) AS aum
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE c.region IN ({region_filter})
      AND f.asset_class IN ({ac_filter})
      AND f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
    GROUP BY c.region ORDER BY aum DESC
""").to_pandas()
st.bar_chart(aum_region.set_index("REGION"))

# ---- AUM by Asset Class (bar chart) ----
st.subheader("AUM by Asset Class")
aum_ac = session.sql(f"""
    SELECT f.asset_class, ROUND(SUM(f.market_value), 0) AS aum
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE c.region IN ({region_filter})
      AND f.asset_class IN ({ac_filter})
      AND f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
    GROUP BY f.asset_class ORDER BY aum DESC
""").to_pandas()
st.bar_chart(aum_ac.set_index("ASSET_CLASS"))

# ---- Performance vs Benchmark (line chart) ----
st.subheader("Avg. Portfolio Return vs Benchmark (Cumulative)")
perf = session.sql(f"""
    SELECT as_of_date,
           ROUND(AVG(portfolio_return), 6) AS avg_port,
           ROUND(AVG(benchmark_return), 6) AS avg_bench
    FROM SNOWCAMP_LAB.MARTS.F_PERFORMANCE_DAILY p
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON p.client_id = c.client_id
    WHERE c.region IN ({region_filter})
    GROUP BY as_of_date ORDER BY as_of_date
""").to_pandas()

if not perf.empty:
    perf["Portfolio"] = (1 + perf["AVG_PORT"]).cumprod() - 1
    perf["Benchmark"] = (1 + perf["AVG_BENCH"]).cumprod() - 1
    st.line_chart(perf[["AS_OF_DATE", "Portfolio", "Benchmark"]].set_index("AS_OF_DATE"))

# ---- Holdings Detail (data table) ----
st.subheader("Top Holdings")
holdings = session.sql(f"""
    SELECT f.portfolio_name, c.client_name, c.region,
           f.security_name, f.isin, f.sector, f.asset_class,
           f.quantity, ROUND(f.market_value, 2) AS market_value,
           ROUND(f.portfolio_weight * 100, 2) AS weight_pct
    FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
    JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
    WHERE c.region IN ({region_filter})
      AND f.asset_class IN ({ac_filter})
      AND f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
    ORDER BY f.market_value DESC LIMIT 100
""").to_pandas()
st.dataframe(holdings, use_container_width=True)

### Deploying as a Standalone Streamlit App

The dashboard above runs inline in this notebook. To deploy it as a **standalone Streamlit app** that other users can access, you can use `CREATE STREAMLIT` pointing to a stage or Git repository.

The lab’s GitHub repo includes a ready-to-deploy app at `streamlit/client_reporting_app.py`.

> **Reference**: [CREATE STREAMLIT](https://docs.snowflake.com/en/sql-reference/sql/create-streamlit) | [Streamlit in Snowflake Overview](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit)

In [ ]:
-- Deploy standalone Streamlit app from the Git repo
-- (requires the Git Repository created in Part 2)

-- List available files in the repo
LS @SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO/branches/main/streamlit/;

-- Create the Streamlit app object
CREATE OR REPLACE STREAMLIT SNOWCAMP_LAB.ANALYTICS.CLIENT_REPORTING_APP
    ROOT_LOCATION  = '@SNOWCAMP_LAB.RAW.SNOWCAMP_GIT_REPO/branches/main/streamlit'
    MAIN_FILE      = 'client_reporting_app.py'
    QUERY_WAREHOUSE = WH_LAB
    TITLE          = 'DWS Client Reporting'
    COMMENT        = 'SnowCamp lab - interactive client reporting dashboard';

-- Open the app URL shown below in a new browser tab
SHOW STREAMLITS IN SCHEMA SNOWCAMP_LAB.ANALYTICS;

---
## Part 4: Internal Marketplace & Horizon Catalog (10 min)

Now let’s **govern and publish** the data product. Snowflake’s **Horizon Catalog** provides tags, classifications, lineage, and access policies. The **Internal Marketplace** lets you publish certified data products that other teams within your organisation can discover and consume.

> **Reference**: [Snowflake Horizon](https://docs.snowflake.com/en/user-guide/governance-horizon) | [Object Tagging](https://docs.snowflake.com/en/user-guide/object-tagging) | [CREATE TAG](https://docs.snowflake.com/en/sql-reference/sql/create-tag) | [Data Listings](https://docs.snowflake.com/en/user-guide/data-marketplace-listings)

In [ ]:
USE SCHEMA SNOWCAMP_LAB.MARTS;

-- Create governance tags
CREATE OR REPLACE TAG DATA_DOMAIN
    ALLOWED_VALUES 'Client Reporting', 'Risk', 'Compliance', 'ESG'
    COMMENT = 'Business domain that owns the data product';

CREATE OR REPLACE TAG CONFIDENTIALITY
    ALLOWED_VALUES 'Public', 'Internal', 'Confidential', 'Restricted'
    COMMENT = 'Data classification level';

CREATE OR REPLACE TAG DATA_STEWARD
    COMMENT = 'Team or person responsible for data quality';

-- Apply tags to mart tables
ALTER TABLE F_POSITIONS_DAILY SET TAG
    DATA_DOMAIN     = 'Client Reporting',
    CONFIDENTIALITY = 'Internal',
    DATA_STEWARD    = 'DWS Data Engineering';

ALTER TABLE F_PERFORMANCE_DAILY SET TAG
    DATA_DOMAIN     = 'Client Reporting',
    CONFIDENTIALITY = 'Internal',
    DATA_STEWARD    = 'DWS Data Engineering';

ALTER TABLE D_PORTFOLIO SET TAG
    DATA_DOMAIN     = 'Client Reporting',
    CONFIDENTIALITY = 'Internal',
    DATA_STEWARD    = 'DWS Data Engineering';

ALTER TABLE D_CLIENT SET TAG
    DATA_DOMAIN     = 'Client Reporting',
    CONFIDENTIALITY = 'Confidential',
    DATA_STEWARD    = 'DWS Client Onboarding';

-- Verify tags
SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('F_POSITIONS_DAILY', 'TABLE'));

In [ ]:
-- Create a share for the client reporting data product
CREATE OR REPLACE SHARE SNOWCAMP_CLIENT_REPORTING_SHARE
    COMMENT = 'Client Reporting Data Product - SnowCamp Lab';

-- Grant access to the mart database and tables
GRANT USAGE ON DATABASE SNOWCAMP_LAB TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT USAGE ON SCHEMA SNOWCAMP_LAB.MARTS TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY    TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.F_PERFORMANCE_DAILY  TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.D_PORTFOLIO          TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;
GRANT SELECT ON TABLE SNOWCAMP_LAB.MARTS.D_CLIENT             TO SHARE SNOWCAMP_CLIENT_REPORTING_SHARE;

SHOW SHARES LIKE 'SNOWCAMP%';

### Publishing to the Internal Marketplace (UI)

To publish your share as a discoverable **Internal Marketplace listing**, use the Snowsight UI:

1. Navigate to **Data Products** → **Private Sharing** in the left sidebar
2. Click **Share** → select the `SNOWCAMP_CLIENT_REPORTING_SHARE`
3. Add a title: *"Client Reporting Mart — Positions & Performance"*
4. Add a description explaining the tables, refresh cadence, and data steward
5. Choose **Internal** as the listing visibility
6. Publish the listing

Consumers within your Snowflake organisation can then **discover, request access, and query** this data product — without any data copying.

> **Reference**: [Creating Listings](https://docs.snowflake.com/en/user-guide/data-marketplace-listings) | [Internal Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace-internal)

---
## Part 5: Platform Administration & Monitoring (15 min)

Finally, let’s look at how DWS would **administer, monitor, and secure** the platform. Snowflake provides rich account-usage views, query insights, and cost-management tools.

> **Reference**: [ACCOUNT_USAGE Schema](https://docs.snowflake.com/en/sql-reference/account-usage) | [Resource Monitors](https://docs.snowflake.com/en/user-guide/resource-monitors) | [Query Insights](https://docs.snowflake.com/en/user-guide/ui-snowsight/activity)

In [ ]:
-- Credit consumption from today's lab session
-- (ACCOUNT_USAGE views have up to 45 min latency; ORGANIZATION_USAGE is near-real-time)
SELECT
    warehouse_name,
    SUM(credits_used)               AS total_credits,
    SUM(credits_used_compute)       AS compute_credits,
    SUM(credits_used_cloud_services) AS cloud_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY total_credits DESC;

In [ ]:
-- Query performance analysis - find the most expensive queries from this lab
SELECT
    query_id,
    query_type,
    warehouse_name,
    execution_status,
    ROUND(total_elapsed_time / 1000, 2) AS elapsed_sec,
    ROUND(bytes_scanned / 1e6, 2)       AS mb_scanned,
    ROUND(compilation_time / 1000, 2)   AS compile_sec,
    ROUND(execution_time / 1000, 2)     AS exec_sec,
    rows_produced,
    query_text
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
ORDER BY total_elapsed_time DESC
LIMIT 15;

In [ ]:
-- Create a resource monitor to alert when credit usage exceeds thresholds
-- In production, this prevents runaway costs
CREATE OR REPLACE RESOURCE MONITOR LAB_MONITOR
    WITH
        CREDIT_QUOTA = 5
        FREQUENCY    = DAILY
        START_TIMESTAMP = IMMEDIATELY
        TRIGGERS
            ON 75 PERCENT DO NOTIFY
            ON 90 PERCENT DO NOTIFY
            ON 100 PERCENT DO SUSPEND;

-- Assign the monitor to the lab warehouse
ALTER WAREHOUSE WH_LAB SET RESOURCE_MONITOR = LAB_MONITOR;

SHOW RESOURCE MONITORS;

### Snowsight UI: Monitoring & Governance Features

Open these pages in Snowsight to explore the built-in monitoring tools:

| Feature | Where to Find It | What It Shows |
|---------|-------------------|--------------|
| **Query Insights** | Activity → Query History | Slow queries, scan efficiency, queuing |
| **Performance Explorer** | Admin → Warehouses → (select WH) | Load patterns, concurrency, auto-scaling |
| **Cost Management** | Admin → Cost Management | Credit usage trends, resource monitors |
| **Trust Center** | Admin → Security → Trust Center | Security posture, policy compliance |
| **Lineage** | Data → (select table) → Lineage tab | Upstream/downstream data flow |

> **Reference**: [Query Insights](https://docs.snowflake.com/en/user-guide/ui-snowsight/activity) | [Performance Explorer](https://docs.snowflake.com/en/user-guide/ui-snowsight/activity-query-insights) | [Cost Management](https://docs.snowflake.com/en/user-guide/cost-management) | [Trust Center](https://docs.snowflake.com/en/user-guide/ui-snowsight/trust-center)

In [ ]:
-- Audit: Who logged in during the lab session?
SELECT
    user_name,
    event_type,
    is_success,
    client_ip,
    first_authentication_factor,
    reported_client_type,
    event_timestamp
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE event_timestamp >= DATEADD('hour', -3, CURRENT_TIMESTAMP())
ORDER BY event_timestamp DESC
LIMIT 20;

---
## Congratulations!

You have built an **end-to-end client-reporting data product** on Snowflake:

| Step | What You Built |
|------|---------------|
| **Synthetic Data** | 6 raw tables mimicking asset-management sources |
| **dbt Transformation** | Staging views → Intermediate tables → Mart fact/dimension tables |
| **Streamlit Dashboard** | Interactive portfolio analytics app with filters & charts |
| **Marketplace Listing** | Governed, tagged, and published data product |
| **Platform Admin** | Credit monitoring, query analysis, and resource monitors |

### Next Steps

- **Explore dbt scheduling**: Set up recurring dbt builds via Snowflake Tasks
- **Add data quality tests**: Use dbt tests (not_null, unique, relationships) for automated validation
- **Connect Tableau**: Point Tableau Desktop/Server at the mart tables for pixel-perfect reporting
- **Try Cortex Code**: Use AI-assisted development to extend models and apps

### Resources

| Topic | Link |
|-------|------|
| Snowflake Notebooks | [docs.snowflake.com/...notebooks](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks) |
| dbt Projects on Snowflake | [docs.snowflake.com/...dbt](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) |
| Streamlit in Snowflake | [docs.snowflake.com/...streamlit](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) |
| Horizon Catalog | [docs.snowflake.com/...horizon](https://docs.snowflake.com/en/user-guide/governance-horizon) |
| OpenFlow | [docs.snowflake.com/...openflow](https://docs.snowflake.com/en/user-guide/data-load-openflow) |
| ACCOUNT_USAGE | [docs.snowflake.com/...account-usage](https://docs.snowflake.com/en/sql-reference/account-usage) |
| Cortex Code | [docs.snowflake.com/...cortex-code](https://docs.snowflake.com/en/user-guide/ui-snowsight/cortex-code) |

> *Thank you for attending DWS SnowCamp!*